# Simulación de Plataforma de Streaming — Parte E

**Taller 2 de Modelos y Simulación**

**Integrantes:** Julian Zapata · Juan José Orrego

---

## Descripción

Este notebook implementa el pipeline completo de simulación de una plataforma de streaming (similar a Netflix/YouTube/Twitch). El sistema modela cómo múltiples usuarios compiten por recursos limitados (ancho de banda, capacidad de procesamiento) y evalúa el riesgo de degradación del servicio (QoS).

### Pipeline:
1. **Parte A — Delphi:** Validación de factores mediante panel de 4 expertos del sector tecnológico
2. **Parte B — Sistema Difuso Mamdani:** Evaluación del riesgo QoS (0–100)
3. **Parte C — Montecarlo:** 5000 simulaciones estocásticas
4. **Parte D — Regresión:** KNN, Random Forest, Decision Tree

In [ ]:
# ============================================================
# Celda 2: Configuración global
# ============================================================
import sys
import os

# Agregar la raíz del proyecto al path para importar trabajo_final
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# Constante global de semilla
RANDOM_SEED = 42

# Directorios relativos al notebook
DATA_DIR = '../data/'
DOCS_DIR = '../docs/'
CONSENSO_PATH = '../data/delphi_consenso.json'
CSV_PATH = '../data/base_simulada_streaming.csv'

# Imports estándar
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print(f'ROOT: {ROOT}')
print(f'RANDOM_SEED: {RANDOM_SEED}')
print(f'Python: {sys.version}')

In [ ]:
# ============================================================
# Celda 3: Parte A — Delphi
# ============================================================
from trabajo_final.delphi import ExpertPanel, DelphiSimulator

print('=' * 60)
print('PARTE A — PROCESO DELPHI')
print('=' * 60)

# Instanciar panel de expertos
panel = ExpertPanel(seed=RANDOM_SEED)
experts = panel.get_experts()

print('\nPanel de expertos:')
for e in experts:
    print(f'  {e.id}: {e.nombre} | {e.cargo} | {e.dependencia}')

# Instanciar simulador
sim = DelphiSimulator(panel, data_dir=DATA_DIR, docs_dir=DOCS_DIR)

print('\n--- Ronda 1: Evaluación inicial ---')
r1 = sim.run_round1()
for fd in r1['factores']:
    st = fd['estadisticos']
    print(f"  {fd['factor']:30s} media={st['media']:.3f}  CV={st['cv']:.3f}")

print('\n--- Ronda 2: Retroalimentación y ajuste ---')
r2 = sim.run_round2(r1)
for fd in r2['factores']:
    st = fd['estadisticos']
    print(f"  {fd['factor']:30s} media={st['media']:.3f}  CV={st['cv']:.3f}")

print('\n--- Ronda 3: Validación final ---')
r3 = sim.run_round3(r2)

print('\nVariables aprobadas:')
for v in r3['variables_aprobadas']:
    st = v['estadisticos_finales']
    pct = v['criterios']['approval_pct']
    print(f"  ✅ {v['factor']:30s} media={st['media']:.3f}  CV={st['cv']:.3f}  aprobación={pct:.1f}%")

if r3['variables_rechazadas']:
    print('\nVariables rechazadas:')
    for v in r3['variables_rechazadas']:
        print(f"  ❌ {v['factor']}")
else:
    print('\n✓ Todas las variables alcanzaron consenso')

print(f'\nArchivos generados:')
print(f'  {DATA_DIR}delphi_ronda1.json')
print(f'  {DATA_DIR}delphi_ronda2.json')
print(f'  {DATA_DIR}delphi_consenso.json')
print(f'  {DOCS_DIR}delphi_informe.md')

In [ ]:
# ============================================================
# Celda 4: Verificación de trazabilidad
# ============================================================
import json

print('=' * 60)
print('VERIFICACIÓN DE TRAZABILIDAD DELPHI')
print('=' * 60)

with open(CONSENSO_PATH, encoding='utf-8') as f:
    consenso = json.load(f)

print(f'\nVariables aprobadas en consenso: {len(consenso["variables_aprobadas"])}')
print(f'Variables rechazadas en consenso: {len(consenso["variables_rechazadas"])}')

print('\nDetalle de variables aprobadas:')
print(f'{"Factor":<30} {"Media":>8} {"Std":>8} {"CV":>8} {"Aprobación":>12}')
print('-' * 70)
for v in consenso['variables_aprobadas']:
    st = v['estadisticos_finales']
    pct = v['criterios']['approval_pct']
    print(f"{v['factor']:<30} {st['media']:>8.4f} {st['std']:>8.4f} {st['cv']:>8.4f} {pct:>11.1f}%")

# Verificar criterios
print('\nVerificación de criterios de consenso:')
print('  Criterio 1: media ≥ 4.0')
print('  Criterio 2: CV ≤ 0.30')
print('  Criterio 3: aprobación ≥ 70%')

all_ok = True
for v in consenso['variables_aprobadas']:
    st = v['estadisticos_finales']
    pct = v['criterios']['approval_pct']
    ok = st['media'] >= 4.0 and st['cv'] <= 0.30 and pct >= 70.0
    status = '✅' if ok else '❌'
    print(f"  {status} {v['factor']}")
    if not ok:
        all_ok = False

print(f'\n{"✓ Trazabilidad verificada" if all_ok else "⚠ Revisar criterios"}')

In [ ]:
# ============================================================
# Celda 5: Parte B — Sistema Difuso Mamdani
# ============================================================
from trabajo_final.fuzzy_system import FuzzySystemBuilder

print('=' * 60)
print('PARTE B — SISTEMA DIFUSO MAMDANI')
print('=' * 60)

# Construir sistema difuso
fs = FuzzySystemBuilder(
    consenso_path=CONSENSO_PATH,
    data_dir=DATA_DIR,
    docs_dir=DOCS_DIR
)
fs.build()
print('\n✓ Sistema difuso construido')
print('  Variables de entrada: usuarios_concurrentes, uso_ancho_banda, latencia_red, capacidad_servidor')
print('  Variable de salida: riesgo_qos [0, 100]')
print('  Defuzzificación: centroide')
print('  Reglas Mamdani: 12')

# Pruebas del sistema difuso
print('\n--- Pruebas del sistema difuso ---')

escenarios = [
    {
        'nombre': 'Escenario ALTO (carga máxima)',
        'valores': {'usuarios_concurrentes': 85.0, 'uso_ancho_banda': 90.0,
                    'latencia_red': 8.0, 'capacidad_servidor': 88.0}
    },
    {
        'nombre': 'Escenario BAJO (carga mínima)',
        'valores': {'usuarios_concurrentes': 10.0, 'uso_ancho_banda': 15.0,
                    'latencia_red': 1.0, 'capacidad_servidor': 20.0}
    },
    {
        'nombre': 'Escenario MEDIO (carga normal)',
        'valores': {'usuarios_concurrentes': 55.0, 'uso_ancho_banda': 60.0,
                    'latencia_red': 5.0, 'capacidad_servidor': 60.0}
    },
    {
        'nombre': 'Escenario LATENCIA ALTA',
        'valores': {'usuarios_concurrentes': 50.0, 'uso_ancho_banda': 50.0,
                    'latencia_red': 9.0, 'capacidad_servidor': 50.0}
    },
]

for esc in escenarios:
    riesgo = fs.evaluar_riesgo(esc['valores'])
    assert 0 <= riesgo <= 100, f'Riesgo fuera de rango: {riesgo}'
    print(f"  {esc['nombre']:40s} → riesgo_qos = {riesgo:.2f}")

print(f'\nArchivos generados:')
print(f'  {DATA_DIR}fuzzy_variables.json')
print(f'  {DATA_DIR}fuzzy_rules.json')
print(f'  {DOCS_DIR}fuzzy_membership_plots/ (5 PNG)')

In [ ]:
# ============================================================
# Celda 6: Parte C — Montecarlo (5000 simulaciones)
# ============================================================
from trabajo_final.montecarlo import MontecarloSimulator

print('=' * 60)
print('PARTE C — SIMULACIÓN MONTECARLO')
print('=' * 60)

print('\nDistribuciones utilizadas:')
print('  usuarios_concurrentes : Beta(α=2, β=3, escala=100)')
print('  uso_ancho_banda        : Normal Truncada(μ=55, σ=20, [0,100])')
print('  latencia_red           : Triangular(mín=0, moda=3, máx=10)')
print('  capacidad_servidor     : Beta(α=3, β=2, escala=100)')

print(f'\nEjecutando 5000 simulaciones con RANDOM_SEED={RANDOM_SEED}...')

mc = MontecarloSimulator(
    fs,
    data_dir=DATA_DIR,
    docs_dir=DOCS_DIR,
    seed=RANDOM_SEED
)
df = mc.run(n_simulaciones=5000)

assert len(df) == 5000, f'Se esperaban 5000 filas, se obtuvieron {len(df)}'
print(f'\n✓ Simulaciones completadas: {len(df)}')

# Estadísticas
stats = mc._statistics
print('\nEstadísticas del riesgo_qos:')
print(f'  Media:          {stats["mean"]:.4f}')
print(f'  Std:            {stats["std"]:.4f}')
print(f'  Mínimo:         {stats["min"]:.4f}')
print(f'  P25:            {stats["p25"]:.4f}')
print(f'  Mediana (P50):  {stats["p50"]:.4f}')
print(f'  P75:            {stats["p75"]:.4f}')
print(f'  P95:            {stats["p95"]:.4f}')
print(f'  Máximo:         {stats["max"]:.4f}')
print(f'  P(riesgo≥70):   {stats["p_riesgo_alto"]:.4f} ({stats["p_riesgo_alto"]:.1%})')

print('\nPrimeras 5 filas del DataFrame:')
print(df.head().to_string())

print(f'\nArchivos generados:')
print(f'  {DATA_DIR}base_simulada_streaming.csv')
print(f'  {DOCS_DIR}montecarlo_histograma_streaming.png')
print(f'  {DOCS_DIR}montecarlo_distribuciones_streaming.md')

In [ ]:
# ============================================================
# Celda 7: Parte D — Regresión
# ============================================================
from trabajo_final.regression import RegressionAnalyzer

print('=' * 60)
print('PARTE D — ANÁLISIS DE REGRESIÓN')
print('=' * 60)

ra = RegressionAnalyzer(
    data_path=CSV_PATH,
    consenso_path=CONSENSO_PATH,
    docs_dir=DOCS_DIR
)

print('\nEntrenando modelos (partición 80/20, random_state=42)...')
metrics = ra.train_and_evaluate()

print('\nMétricas de evaluación:')
model_display = {
    'knn': 'KNN (k=5)',
    'random_forest': 'Random Forest (100 árboles)',
    'decision_tree': 'Decision Tree',
}
print(f'{"Modelo":<35} {"MAE":>8} {"RMSE":>8} {"R²":>8}')
print('-' * 62)
best_name = max(metrics, key=lambda k: metrics[k]['r2'])
for name, m in metrics.items():
    marker = ' ✓' if name == best_name else '  '
    print(f"{model_display.get(name, name):<35}{marker} {m['mae']:>8.4f} {m['rmse']:>8.4f} {m['r2']:>8.4f}")

print(f'\nMejor modelo: {model_display.get(best_name, best_name)} (R²={metrics[best_name]["r2"]:.4f})')

# Correlación de Pearson
corr = ra.calculate_pearson_correlation()
print(f'\nCorrelación de Pearson (mejor modelo vs. difuso): r = {corr:.4f}')

# Importancia de variables
importances = ra.get_feature_importance()
print('\nImportancia de variables (Random Forest):')
rf_imp = sorted(importances['random_forest'].items(), key=lambda x: x[1], reverse=True)
for var, val in rf_imp:
    print(f'  {var:<30} {val:.4f} ({val*100:.2f}%)')

# Generar reportes
print('\nGenerando reportes...')
ra.generate_scatter_plot()
ra.generate_comparative_report(metrics)
ra.generate_importance_report()
ra.generate_comparative_analysis(metrics, corr)

print(f'\nArchivos generados:')
print(f'  {DOCS_DIR}regression_comparativa_streaming.md')
print(f'  {DOCS_DIR}regression_importancia_streaming.md')
print(f'  {DOCS_DIR}analisis_comparativo_streaming.md')
print(f'  {DOCS_DIR}comparativo_difuso_vs_prediccion_streaming.png')

In [ ]:
# ============================================================
# Celda 8: Resumen final
# ============================================================
import os

print('=' * 60)
print('RESUMEN FINAL — PARTE E')
print('Simulación de Plataforma de Streaming')
print('=' * 60)

print('\n📋 PARTE A — DELPHI')
print(f'  Variables aprobadas: {[v["factor"] for v in r3["variables_aprobadas"]]}')
print(f'  Variables rechazadas: {[v["factor"] for v in r3["variables_rechazadas"]]}')

print('\n🔷 PARTE B — SISTEMA DIFUSO')
riesgo_alto = fs.evaluar_riesgo({
    'usuarios_concurrentes': 85.0,
    'uso_ancho_banda': 90.0,
    'latencia_red': 8.0,
    'capacidad_servidor': 88.0
})
print(f'  Riesgo QoS (escenario alto): {riesgo_alto:.2f}')
print(f'  Reglas Mamdani: 12')
print(f'  Defuzzificación: centroide')

print('\n🎲 PARTE C — MONTECARLO (5000 simulaciones)')
print(f'  Media riesgo_qos:    {stats["mean"]:.4f}')
print(f'  Mediana riesgo_qos:  {stats["p50"]:.4f}')
print(f'  P(riesgo_qos ≥ 70): {stats["p_riesgo_alto"]:.4f} ({stats["p_riesgo_alto"]:.1%})')

print('\n📊 PARTE D — REGRESIÓN')
for name, m in metrics.items():
    marker = ' ← mejor' if name == best_name else ''
    print(f'  {model_display.get(name, name):<35} R²={m["r2"]:.4f}{marker}')
print(f'  Correlación Pearson: r = {corr:.4f}')

print('\n📁 ARCHIVOS GENERADOS')
archivos = [
    '../data/delphi_ronda1.json',
    '../data/delphi_ronda2.json',
    '../data/delphi_consenso.json',
    '../data/fuzzy_variables.json',
    '../data/fuzzy_rules.json',
    '../data/base_simulada_streaming.csv',
    '../docs/delphi_informe.md',
    '../docs/fuzzy_membership_plots/usuarios_concurrentes.png',
    '../docs/fuzzy_membership_plots/uso_ancho_banda.png',
    '../docs/fuzzy_membership_plots/latencia_red.png',
    '../docs/fuzzy_membership_plots/capacidad_servidor.png',
    '../docs/fuzzy_membership_plots/riesgo_qos.png',
    '../docs/montecarlo_histograma_streaming.png',
    '../docs/montecarlo_distribuciones_streaming.md',
    '../docs/regression_comparativa_streaming.md',
    '../docs/regression_importancia_streaming.md',
    '../docs/analisis_comparativo_streaming.md',
    '../docs/comparativo_difuso_vs_prediccion_streaming.png',
]

nb_dir = os.path.dirname(os.path.abspath('streaming_completo.ipynb'))
for archivo in archivos:
    ruta = os.path.join(nb_dir, archivo)
    existe = os.path.exists(ruta)
    status = '✅' if existe else '❌'
    print(f'  {status} {archivo}')

print('\n✓ Pipeline completo ejecutado exitosamente')